# B4 - Connect-Four with a ResNet trunk

Same AlphaZero machinery, a bigger **spatial** board (6x7). Now the network uses the
**ResNet trunk from A2** (`trunk="resnet"`) because the board has 2D structure worth
convolving over - exactly raccoon's design choice. Connect-Four is too big to solve
exactly, so we judge strength by **match play vs baselines** (raccoon's arena idea).

In [1]:
import numpy as np, torch
from azl.games.connect4 import Connect4
from azl.network import net_for_game
from azl.trainer import Coach, CoachConfig
from azl.evaluate import mcts_player, play_match, random_player

torch.manual_seed(0)
net = net_for_game(Connect4)        # picks trunk="resnet" automatically
print("trunk:", net.trunk_kind, "| input shape:", net.input_shape)

coach = Coach(net=net, game_cls=Connect4,
              config=CoachConfig(games_per_iter=12, num_simulations=40, temp_threshold=6, training_steps=40), seed=0)
coach.train(10)

trunk: resnet | input shape: (2, 6, 7)


iter   1  policy 1.648  value 0.111  entropy 1.524  moves 31.5  p0_ret +0.67  (47.74s)


iter   2  policy 1.649  value 0.102  entropy 1.428  moves 31.6  p0_ret +0.42  (61.0s)


iter   3  policy 1.565  value 0.127  entropy 1.380  moves 22.7  p0_ret +0.08  (53.59s)


iter   4  policy 1.559  value 0.142  entropy 1.413  moves 22.9  p0_ret +0.17  (45.66s)


...

iter   7  policy 1.489  value 0.187  entropy 1.083  moves 15.8  p0_ret +0.17  (24.43s)


iter   8  policy 1.460  value 0.182  entropy 1.155  moves 18.1  p0_ret +0.50  (30.28s)


iter   9  policy 1.453  value 0.195  entropy 1.104  moves 17.1  p0_ret +0.17  (22.09s)


iter  10  policy 1.438  value 0.210  entropy 1.080  moves 19.8  p0_ret +0.50  (49.9s)


[{'policy_loss': 1.6478645026683807,
  'value_loss': 0.11107927043922246,
  'iteration': 1,
  'buffer': 378,
  'avg_moves': 31.5,
  'avg_visit_entropy': 1.524146150199468,
  'p0_return': 0.6666666666666666,
  'seconds': 47.74},
 {'policy_loss': 1.6491715103387832,
  'value_loss': 0.10218501170165836,
  'iteration': 2,
  'buffer': 757,
  'avg_moves': 31.583333333333332,
  'avg_visit_entropy': 1.4282149715826857,
  'p0_return': 0.4166666666666667,
  'seconds': 61.0},
 {'policy_loss': 1.5653052985668183,
  'value_loss': 0.12720532361418008,
  'iteration': 3,
  'buffer': 1029,
  'avg_moves': 22.666666666666668,
  'avg_visit_entropy': 1.38047854308879,
  'p0_return': 0.08333333333333333,
  'seconds': 53.59},
 {'policy_loss': 1.559490829706192,
  'value_loss': 0.14197272323071958,
  'iteration': 4,
  'buffer': 1304,
  'avg_moves': 22.916666666666668,
  'avg_visit_entropy': 1.4129696377493504,
  'p0_return': 0.16666666666666666,
  'seconds': 45.66},
 {'policy_loss': 1.5212586790323257,
  'val

In [2]:
rng = np.random.default_rng(1)
before = play_match(Connect4, random_player(np.random.default_rng(2)), random_player(rng), num_games=40, rng=rng)
res = play_match(Connect4, mcts_player(net, 80, rng=rng), random_player(rng), num_games=40, rng=rng)
print("random vs random (baseline):", before)
print("trained agent vs random    :", res, " <- should win clearly")

random vs random (baseline): {'num_games': 40, 'a_wins': 21, 'draws': 0, 'a_losses': 19, 'a_return_mean': 0.05}
trained agent vs random    : {'num_games': 40, 'a_wins': 39, 'draws': 1, 'a_losses': 0, 'a_return_mean': 0.975}  <- should win clearly


In [3]:
# Watch the agent play itself; print the final board.
from azl.mcts import advance_through_chance
rng = np.random.default_rng(3)
play = mcts_player(net, 80, rng=rng)
s = advance_through_chance(Connect4.start(), rng)
while not s.is_terminal():
    s = s.apply_action(play(s))
print(s.render())
print("returns:", s.returns())

. . . . . . .
. O . O . . .
. O X O . . .
X X O X . . .
O X O X . . O
O X O X X X X
0 1 2 3 4 5 6
returns: [1.0, -1.0]


### Things to tweak
- More `iterations` / `num_simulations` -> stronger play (Connect-Four needs more
  than tic-tac-toe).
- `channels`, `num_blocks` in `net_for_game(Connect4, channels=..., num_blocks=...)`.

### Maps to raccoon
The `resnet` trunk here is `azl.foundations.blocks.ConvTrunk` - the same residual
blocks from A2 - feeding policy/value heads, which is structurally raccoon's
`RaccoonNet` (just 2x12 board vs 6x7, and 1352 actions vs 7).